# 🚧 Estruturação de Dados — DNIT × PRF (Região Norte)

**Notebook único de limpeza e estruturação** dos dados do **DNIT** (condição das rodovias, ICM)
e da **PRF** (acidentes), deixando-os prontos para **análise exploratória** e **treinamento de modelos**.

> **Escopo desta entrega:** *estruturar* os dados. A análise exploratória, os modelos preditivos e o
> deploy (Streamlit) serão construídos depois, sobre as bases geradas aqui.

**Regras (pedidos do Athayde):**
1. Padrão de colunas fixo e ordenado.
2. Abrangência: **estados do Norte** → apenas as **TOP 10 BRs com mais linhas**.
3. Foco em **ter dados** (maximizar volume).
4. Eliminar colunas inúteis (latitude, longitude, observação, mês…).
5. PRF restrita às **mesmas TOP 10 BRs** e ao **mesmo período do DNIT**.
6. Um único notebook, no repositório **PredicaoAcidentesPRF**.

> ⚠️ **Escala do ICM a confirmar:** há divergência entre fontes (maior=melhor × maior=pior).
> Os valores de ICM são mantidos **como no original**; a interpretação fica para a etapa de análise.


## ⚙️ Etapa 0 — Configuração

In [1]:
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)

# Caminhos
RAW = Path("../data/raw")
PROC = Path("../data/processed")
PROC.mkdir(parents=True, exist_ok=True)

DNIT_CONSOLIDADO = RAW / "dnit_consolidado.csv"   # base do DNIT (já consolidada)
KAGGLE_DATASET = "lucasandroliveira/dnit-prf-norte"  # fonte do DNIT no Kaggle (fallback)

def get_dnit_consolidado():
    # Caminho do dnit_consolidado.csv: usa o local se existir; senão baixa do Kaggle.
    if DNIT_CONSOLIDADO.exists():
        print("DNIT: usando arquivo local ->", DNIT_CONSOLIDADO)
        return DNIT_CONSOLIDADO
    import kagglehub
    print("DNIT: baixando do Kaggle ->", KAGGLE_DATASET)
    base_kaggle = Path(kagglehub.dataset_download(KAGGLE_DATASET))
    # procura recursivamente (resiliente a subpastas na estrutura do dataset)
    achados = list(base_kaggle.rglob("dnit_consolidado.csv"))
    if not achados:
        raise FileNotFoundError("dnit_consolidado.csv não encontrado no dataset Kaggle")
    return achados[0]

# Escopo geográfico e temporal
UFS_NORTE = ["AC", "AM", "AP", "PA", "RO", "RR", "TO"]
ANOS = [2024, 2025, 2026]            # janela de cruzamento DNIT × PRF
N_TOP_BRS = 10

# PRF — Dados Abertos (datatran / agrupados por pessoa). Baixados via gdown se faltarem.
PRF_DRIVE_IDS = {
    2024: "14lVfqdoE2gxDliaKZu7K9Mx6847maPtl",
    2025: "1-Gp9S-ALO0D1nT8S_OKoC8xlW7BY8F82",
    2026: "1B_rvx1kPHuFowP5rs84ZO-57gN8yUs85",
}

def num_br(serie):
    """Extrai o número da BR (ex.: 'BR-364' -> '364', 364.0 -> '364')."""
    return (serie.astype(str).str.extract(r"(\d{1,3})")[0]
            .str.zfill(3))

print("Configuração pronta. Norte:", UFS_NORTE, "| Anos:", ANOS)


Configuração pronta. Norte: ['AC', 'AM', 'AP', 'PA', 'RO', 'RR', 'TO'] | Anos: [2024, 2025, 2026]


## 🛣️ Etapa 1 — DNIT (condição das rodovias)

A base `dnit_consolidado.csv` já reúne os levantamentos do DNIT. Aqui apenas **estruturamos**:
filtramos Norte, escolhemos as TOP 10 BRs com mais linhas, recortamos a janela de anos,
removemos colunas inúteis e deduplicamos cada trecho pela avaliação **mais recente**.

In [2]:
dnit = pd.read_csv(get_dnit_consolidado(), sep=";", low_memory=False)
print("DNIT consolidado bruto:", dnit.shape)

dnit["uf"] = dnit["uf"].astype(str).str.upper().str.strip()
dnit["br"] = num_br(dnit["rodovia"])
dnit["ano"] = pd.to_numeric(dnit["ano"], errors="coerce")

# Filtro Norte + janela de anos
dnit = dnit[dnit["uf"].isin(UFS_NORTE) & dnit["ano"].isin(ANOS)]
dnit = dnit.dropna(subset=["br", "km"])

# TOP 10 BRs com MAIS LINHAS (pedido #2)
TOP_BRS = dnit["br"].value_counts().head(N_TOP_BRS).index.tolist()
print("TOP 10 BRs (por nº de linhas):", TOP_BRS)
dnit = dnit[dnit["br"].isin(TOP_BRS)]


DNIT: usando arquivo local -> ..\data\raw\dnit_consolidado.csv


DNIT consolidado bruto: (3439793, 25)


TOP 10 BRs (por nº de linhas): ['364', '230', '174', '010', '153', '156', '317', '319', '242', '155']


In [3]:
# Colunas inúteis a remover (pedido #4): coordenadas, observação, mês (já há data/ano)
DROP_DNIT = ["latitude", "longitude", "observacao", "mes", "rodovia",
             "id_malha", "contrato", "icm_unificado", "_arquivo_origem"]
dnit = dnit.drop(columns=[c for c in DROP_DNIT if c in dnit.columns])

# Ordem padrão das colunas (pedido #1)
ORDEM_DNIT = ["uf", "br", "km", "km_inicial", "km_final", "extensao", "sentido",
              "ano", "data_aval", "superficie", "num_faixas",
              "cond_pavimento", "cond_conservacao", "cond_pista",
              "icc", "icp", "icm"]
dnit = dnit[[c for c in ORDEM_DNIT if c in dnit.columns]]

# Dedup: um registro por trecho (uf, br, km), mantendo a avaliação mais recente
dnit["_dt"] = pd.to_datetime(dnit.get("data_aval"), errors="coerce", dayfirst=True)
dnit = (dnit.sort_values(["ano", "_dt"])
            .drop_duplicates(subset=["uf", "br", "km"], keep="last")
            .drop(columns="_dt")
            .reset_index(drop=True))

print("DNIT estruturado:", dnit.shape)
print("Cobertura ICM:", f"{dnit['icm'].notna().mean()*100:.1f}%")
display(dnit.head())


DNIT estruturado: (20036, 17)
Cobertura ICM: 89.4%


,uf,br,km,km_inicial,km_final,extensao,sentido,ano,data_aval,superficie,num_faixas,cond_pavimento,cond_conservacao,cond_pista,icc,icp,icm
0,TO,010,106.5,106.0,107.0,1.0,Crescente,2024.0,2024-04-10 00:00:00,NaN,NaN,NaN,X,NaN,25.0,0.0,7.5
1,TO,010,107.5,107.0,108.0,1.0,Crescente,2024.0,2024-04-10 00:00:00,NaN,NaN,NaN,X,NaN,25.0,0.0,7.5
2,TO,010,108.5,108.0,109.0,1.0,Crescente,2024.0,2024-04-10 00:00:00,NaN,NaN,NaN,X,NaN,25.0,0.0,7.5
3,TO,010,109.5,109.0,110.0,1.0,Crescente,2024.0,2024-04-10 00:00:00,NaN,NaN,NaN,X,NaN,25.0,0.0,7.5
4,TO,010,110.5,110.0,111.0,1.0,Crescente,2024.0,2024-04-10 00:00:00,NaN,NaN,NaN,X,NaN,25.0,0.0,7.5


In [4]:
saida_dnit = PROC / "dnit_estruturado.csv"
dnit.to_csv(saida_dnit, index=False, sep=";", encoding="utf-8-sig")
print("Salvo:", saida_dnit.resolve(), f"({saida_dnit.stat().st_size/1e6:.1f} MB)")


Salvo: C:\Users\Lucas\Desktop\CD2\PredicaoAcidentesPRF\data\processed\dnit_estruturado.csv (1.6 MB)


## 🚑 Etapa 2 — PRF (acidentes)

Usamos os arquivos **nacionais** dos Dados Abertos da PRF (formato *datatran* / agrupados por pessoa),
2024–2026. Filtramos para o Norte e para as **mesmas TOP 10 BRs** do DNIT.

In [5]:
def baixar_prf(ano, fid):
    destino = RAW / f"acidentes{ano}.csv"
    if destino.exists():
        return destino
    import gdown, zipfile
    tmp = RAW / f"prf_{ano}.zip"
    gdown.download(id=fid, output=str(tmp), quiet=False)
    with zipfile.ZipFile(tmp) as zf:
        zf.extractall(RAW)
    tmp.unlink(missing_ok=True)
    return destino

def ler_prf(caminho):
    for sep in (";", ","):
        try:
            d = pd.read_csv(caminho, sep=sep, encoding="latin-1", low_memory=False)
            if d.shape[1] > 5:
                return d
        except Exception:
            pass
    raise RuntimeError(f"Falha ao ler {caminho}")

partes = []
for ano, fid in PRF_DRIVE_IDS.items():
    caminho = baixar_prf(ano, fid)
    partes.append(ler_prf(caminho))
prf = pd.concat(partes, ignore_index=True)
print("PRF nacional (2024-2026):", prf.shape)


PRF nacional (2024-2026): (455060, 35)


In [6]:
prf["uf"] = prf["uf"].astype(str).str.upper().str.strip()
prf["br"] = num_br(prf["br"])

# Mesmo recorte do DNIT: Norte + TOP 10 BRs
prf = prf[prf["uf"].isin(UFS_NORTE) & prf["br"].isin(TOP_BRS)].copy()

# Colunas inúteis para o objetivo (coordenadas + administrativas)
DROP_PRF = ["latitude", "longitude", "regional", "delegacia", "uop"]
prf = prf.drop(columns=[c for c in DROP_PRF if c in prf.columns])

# Tipos
prf["km"] = pd.to_numeric(prf["km"].astype(str).str.replace(",", ".", regex=False), errors="coerce")
prf["data_inversa"] = pd.to_datetime(prf["data_inversa"], errors="coerce", format="%Y-%m-%d")
prf["ano"] = prf["data_inversa"].dt.year

prf = prf.dropna(subset=["br", "km"]).reset_index(drop=True)
print("PRF estruturada:", prf.shape)
print("Por ano:", prf["ano"].value_counts().sort_index().to_dict())
print("Por BR:", prf["br"].value_counts().to_dict())
display(prf.head())


PRF estruturada: (21257, 31)
Por ano: {2024: 9181, 2025: 8897, 2026: 3179}
Por BR: {'364': 9375, '153': 3776, '230': 2101, '010': 1844, '174': 1577, '319': 1311, '155': 463, '156': 368, '317': 281, '242': 161}


,id,pesid,data_inversa,dia_semana,horario,uf,br,km,municipio,causa_acidente,tipo_acidente,classificacao_acidente,fase_dia,sentido_via,condicao_metereologica,tipo_pista,tracado_via,uso_solo,id_veiculo,tipo_veiculo,marca,ano_fabricacao_veiculo,tipo_envolvido,estado_fisico,idade,sexo,ilesos,feridos_leves,feridos_graves,mortos,ano
0,572017.0,1274482,2024-01-01,segunda-feira,17:30:00,PA,010,83.3,ULIANOPOLIS,Demais falhas mecânicas ou elétricas,Saída de leito carroçável,Com Vítimas Feridas,Pleno dia,Decrescente,Céu Claro,Simples,Reta,Não,1021918,Caminhão-trator,VOLVO/FH 540 6X4T,2022,Condutor,Lesões Leves,61,Masculino,0,1,0,0,2024
1,572251.0,1270323,2024-01-02,terça-feira,22:00:00,TO,153,511.1,PARAISO DO TOCANTINS,Transitar na contramão,Colisão frontal,Com Vítimas Feridas,Plena Noite,Decrescente,Garoa/Chuvisco,Simples,Reta,Não,1019077,Caminhão-trator,SCANIA/R 440 A6X2,2011,Condutor,Ileso,35,Masculino,1,0,0,0,2024
2,572374.0,1270863,2024-01-03,quarta-feira,13:26:00,TO,153,143.0,ARAGUAINA,Transitar na contramão,Colisão com objeto,Com Vítimas Fatais,Pleno dia,Decrescente,Nublado,Múltipla,Reta,Sim,1019436,Caminhão-trator,IVECO/STRALIS 800S48TZ,2021,Condutor,Ileso,37,Masculino,1,0,0,0,2024
3,572480.0,1271975,2024-01-03,quarta-feira,21:50:00,TO,153,392.0,RIO DOS BOIS,Ausência de reação do condutor,Colisão traseira,Com Vítimas Feridas,Plena Noite,Decrescente,Céu Claro,Simples,Reta,Sim,1020209,Caminhão-trator,M.BENZ/ACTROS 2651LS6X4,2018,Condutor,Ileso,24,Masculino,1,0,0,0,2024
4,572480.0,1271976,2024-01-03,quarta-feira,21:50:00,TO,153,392.0,RIO DOS BOIS,Ausência de reação do condutor,Colisão traseira,Com Vítimas Feridas,Plena Noite,Decrescente,Céu Claro,Simples,Reta,Sim,1020209,Caminhão-trator,M.BENZ/ACTROS 2651LS6X4,2018,Passageiro,Ileso,28,Feminino,1,0,0,0,2024


In [7]:
saida_prf = PROC / "prf_estruturado.csv"
prf.to_csv(saida_prf, index=False, sep=";", encoding="utf-8-sig")
print("Salvo:", saida_prf.resolve(), f"({saida_prf.stat().st_size/1e6:.1f} MB)")


Salvo: C:\Users\Lucas\Desktop\CD2\PredicaoAcidentesPRF\data\processed\prf_estruturado.csv (6.2 MB)


## 🔗 Etapa 3 — Base cruzada (DNIT × PRF) para modelagem

Cada acidente recebe o ICM do trecho correspondente. **Chave: `uf + br + km arredondado`**
(o DNIT já está deduplicado pela avaliação mais recente de cada trecho).

In [8]:
dnit_key = dnit.copy()
dnit_key["km_join"] = dnit_key["km"].round().astype("Int64")
dnit_key = (dnit_key.drop_duplicates(subset=["uf", "br", "km_join"], keep="last")
                    [["uf", "br", "km_join", "icc", "icp", "icm",
                      "cond_pavimento", "cond_conservacao", "cond_pista"]])

prf_key = prf.copy()
prf_key["km_join"] = prf_key["km"].round().astype("Int64")

base = prf_key.merge(dnit_key, on=["uf", "br", "km_join"], how="left", suffixes=("", "_dnit"))
cobertura = base["icm"].notna().mean() * 100
print("Base cruzada:", base.shape, f"| acidentes com ICM casado: {cobertura:.1f}%")

saida_base = PROC / "base_modelagem.csv"
base.to_csv(saida_base, index=False, sep=";", encoding="utf-8-sig")
print("Salvo:", saida_base.resolve(), f"({saida_base.stat().st_size/1e6:.1f} MB)")
display(base.head())


Base cruzada: (21257, 38) | acidentes com ICM casado: 94.8%


Salvo: C:\Users\Lucas\Desktop\CD2\PredicaoAcidentesPRF\data\processed\base_modelagem.csv (6.6 MB)


,id,pesid,data_inversa,dia_semana,horario,uf,br,km,municipio,causa_acidente,tipo_acidente,classificacao_acidente,fase_dia,sentido_via,condicao_metereologica,tipo_pista,tracado_via,uso_solo,id_veiculo,tipo_veiculo,marca,ano_fabricacao_veiculo,tipo_envolvido,estado_fisico,idade,sexo,ilesos,feridos_leves,feridos_graves,mortos,ano,km_join,icc,icp,icm,cond_pavimento,cond_conservacao,cond_pista
0,572017.0,1274482,2024-01-01,segunda-feira,17:30:00,PA,010,83.3,ULIANOPOLIS,Demais falhas mecânicas ou elétricas,Saída de leito carroçável,Com Vítimas Feridas,Pleno dia,Decrescente,Céu Claro,Simples,Reta,Não,1021918,Caminhão-trator,VOLVO/FH 540 6X4T,2022,Condutor,Lesões Leves,61,Masculino,0,1,0,0,2024,83,41.25,20.0,26.375,NaN,NaN,NaN
1,572251.0,1270323,2024-01-02,terça-feira,22:00:00,TO,153,511.1,PARAISO DO TOCANTINS,Transitar na contramão,Colisão frontal,Com Vítimas Feridas,Plena Noite,Decrescente,Garoa/Chuvisco,Simples,Reta,Não,1019077,Caminhão-trator,SCANIA/R 440 A6X2,2011,Condutor,Ileso,35,Masculino,1,0,0,0,2024,511,0.00,0.0,0.000,NaN,NaN,NaN
2,572374.0,1270863,2024-01-03,quarta-feira,13:26:00,TO,153,143.0,ARAGUAINA,Transitar na contramão,Colisão com objeto,Com Vítimas Fatais,Pleno dia,Decrescente,Nublado,Múltipla,Reta,Sim,1019436,Caminhão-trator,IVECO/STRALIS 800S48TZ,2021,Condutor,Ileso,37,Masculino,1,0,0,0,2024,143,0.00,0.0,0.000,NaN,NaN,NaN
3,572480.0,1271975,2024-01-03,quarta-feira,21:50:00,TO,153,392.0,RIO DOS BOIS,Ausência de reação do condutor,Colisão traseira,Com Vítimas Feridas,Plena Noite,Decrescente,Céu Claro,Simples,Reta,Sim,1020209,Caminhão-trator,M.BENZ/ACTROS 2651LS6X4,2018,Condutor,Ileso,24,Masculino,1,0,0,0,2024,392,6.25,0.0,1.875,NaN,NaN,NaN
4,572480.0,1271976,2024-01-03,quarta-feira,21:50:00,TO,153,392.0,RIO DOS BOIS,Ausência de reação do condutor,Colisão traseira,Com Vítimas Feridas,Plena Noite,Decrescente,Céu Claro,Simples,Reta,Sim,1020209,Caminhão-trator,M.BENZ/ACTROS 2651LS6X4,2018,Passageiro,Ileso,28,Feminino,1,0,0,0,2024,392,6.25,0.0,1.875,NaN,NaN,NaN


## ✅ Saídas e próximos passos

Geradas em `data/processed/`:
- **`dnit_estruturado.csv`** — DNIT do Norte (TOP 10 BRs), 1 linha por trecho, avaliação mais recente.
- **`prf_estruturado.csv`** — acidentes PRF do Norte nas mesmas BRs (2024–2026).
- **`base_modelagem.csv`** — acidentes + ICM do trecho (pronto para EDA e modelos).

**A cargo do Athayde (próximas etapas):**
1. Análise exploratória sobre as bases acima.
2. Definição do alvo (ex.: gravidade/feridos/mortos) e *feature engineering* adicional.
3. Treinamento e avaliação dos modelos preditivos.
4. Deploy via **Streamlit**.

> ⚠️ Antes de interpretar o ICM no modelo, **confirmar a direção da escala** (maior = melhor ou pior).
